<a href="https://colab.research.google.com/github/pesco13/QI452-Python-Quimica/blob/main/notebooks/06_Estructuras_3D_y_conformeros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 06 · Estructuras 3D y conformeros

**Objetivos**
* Generar una geometría 3D a partir de un SMILES.
* Optimizarla con un campo de fuerza (mecánica molecular, MMFF94).
* Comparar la energía de varios conformeros y visualizarlos.

In [ ]:
!pip install -q rdkit py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem
import py3Dmol

## 1. De 2D a 3D

RDKit agrega los hidrógenos, genera coordenadas 3D (ETKDG) y las ajusta con el campo de fuerza MMFF94, que trata los enlaces como resortes.

In [ ]:
mol = Chem.AddHs(Chem.MolFromSmiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O"))   # ibuprofeno
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

vista = py3Dmol.view(width=450, height=350)
vista.addModel(Chem.MolToMolBlock(mol), "mol")
vista.setStyle({"stick": {}})
vista.zoomTo()
vista.show()

## 2. Conformeros

Al girar los enlaces simples se obtienen distintas formas 3D de la misma molécula. Se generan 20 y se comparan sus energías.

In [ ]:
confs = AllChem.EmbedMultipleConfs(mol, numConfs=20, randomSeed=7)
resultado = AllChem.MMFFOptimizeMoleculeConfs(mol)   # lista de (convergió, energía)
energias = [e for (_, e) in resultado]
emin = min(energias)
relativas = sorted((e - emin, i) for i, e in enumerate(energias))
for de, i in relativas[:5]:
    print(f"Conformero {i:2d}: ΔE = {de:5.2f} kcal/mol")

## 3. Ver el más estable

In [ ]:
mejor = relativas[0][1]
v = py3Dmol.view(width=450, height=350)
v.addModel(Chem.MolToMolBlock(mol, confId=mejor), "mol")
v.setStyle({"stick": {}}); v.zoomTo(); v.show()

## 4. Guardar en formato XYZ

El formato XYZ (símbolo y coordenadas en Å) es la entrada de muchos programas de química cuántica, como el del notebook 07.

In [ ]:
xyz = Chem.MolToXYZBlock(mol, confId=mejor)
print(xyz[:300])
open("ibuprofeno.xyz", "w").write(xyz)

## Ejercicios

**1.** Para el butano (`CCCC`), genere 10 conformeros y encuentre la diferencia de energía entre el más estable (anti) y el siguiente (gauche).

<details><summary><b>Ver solución</b></summary>

```python
b = Chem.AddHs(Chem.MolFromSmiles("CCCC"))
AllChem.EmbedMultipleConfs(b, 10, randomSeed=1)
E = [e for _, e in AllChem.MMFFOptimizeMoleculeConfs(b)]
print(sorted(round(e - min(E), 2) for e in E))
# Aparecen dos grupos de energías: anti (0) y gauche (≈ 0.8 kcal/mol con MMFF94).
```
</details>

**2.** ¿Por qué un campo de fuerza no puede describir la ruptura de un enlace?

<details><summary><b>Ver solución</b></summary>

```python
# Porque trata los enlaces como resortes con una conectividad fija: no describe electrones,
# así que no puede formar ni romper enlaces. Para eso se necesita química cuántica (notebook 07).
```
</details>